In [1]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

from IPython.display import Video
from IPython.display import Audio as play_audio
from torchvision.utils import make_grid
from torchcodec.encoders import VideoEncoder

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

import matplotlib.pyplot as plt

import sys
sys.path.append('../../../../')

In [2]:
from pathlib import Path
import math
import time
import random
import datetime
from functools import partial
from collections import OrderedDict

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image

%load_ext autoreload
%autoreload 2

from computer_vision.video_mae.finetune_parameter_parser import parser
from computer_vision.video_mae.dataset.datasets import VideoClsDataset

from computer_vision.torch_video.utils.plotting import plot_all
from computer_vision.torch_video.utils.progress import save_metrics

from computer_vision.video_mae.dataset.mixup import Mixup
from computer_vision.video_mae.models.modeling_finetune import vit_small_patch16_224
from computer_vision.video_mae.utils import multiple_samples_collate, seed_worker, load_state_dict, get_world_size, cosine_scheduler,\
MetricLogger, SmoothedValue, get_grad_norm, save_checkpoint, form_stats
from computer_vision.video_mae.models.load_pretrain import load_pretrained_model
from computer_vision.video_mae.optim_factory import LayerDecayValueAssigner, create_optimizer
from computer_vision.video_mae.models.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from computer_vision.video_mae.engine_for_finetuning import train_one_epoch, validation_one_epoch, accuracy

In [3]:
data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
train_annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask/trainlist01.txt'
val_annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask/vallist01.txt'

mini_train=True
if not mini_train:
    pretrain_path=Path('D:/results/ucf101/video_mae/distilled_small/vit_s_k710_dl_from_giant.pth') 
    output_dirpath=Path('D:/results/ucf101/video_mae/finetune_distilled_small') 
    arguments= f"""--data_root {root} --train_data_path {train_annotation_path} --val_data_path {val_annotation_path} 
    --output_dir {output_dirpath} --finetune {pretrain_path} 
    --data_set UCF101 --nb_classes 101  --batch_size 3 --input_size 224 --cutmix 1. --mixup 0.8
    --short_side_size 224 --num_frames 16 --sampling_rate 4 --num_sample 2 --num_workers 0 --opt adamw
    --cos_attn --lr 1e-3 --drop_path 0.3 --clip_grad 5.0 --layer_decay 0.9 --opt_betas 0.9 0.999 --weight_decay 0.1
    --test_num_segment 5 --test_num_crop 3 --dist_eval
    --warmup_epochs 5 --epochs 35  --print_freq 20 --device cuda --time 12 --resume
    """ # --use-cutmix-mixup
else:
    pretrain_path=Path('D:/results/ucf101/video_mae/distilled_small/vit_s_k710_dl_from_giant.pth') 
    output_dirpath=Path('D:/results/ucf101/video_mae/mini_finetune_distilled_small') 
    arguments= f"""--data_root {root} --train_data_path {train_annotation_path} --val_data_path {val_annotation_path} 
    --output_dir {output_dirpath}  --finetune {pretrain_path} 
    --data_set UCF101 --nb_classes 101  --batch_size 3 --input_size 224 --cutmix 1. --mixup 0.8
    --short_side_size 224 --num_frames 16 --sampling_rate 4 --num_sample 2 --num_workers 0 --opt adamw
    --cos_attn --lr 1e-3 --drop_path 0.3 --clip_grad 5.0 --layer_decay 0.9 --opt_betas 0.9 0.999 --weight_decay 0.1
    --test_num_segment 5 --test_num_crop 3 --dist_eval 
    --warmup_epochs 5 --print_freq 20 --epochs 35  --device cuda --resume
    --n_steps 24 --n_epochs 4 --time 0.5
    """ 

known_args, _=parser.parse_known_args(args=arguments.split())
if known_args.enable_deepspeed:
    parser=deepspeed.add_config_arguments(parser)
    ds_init=deepspeed.initialize
else: ds_init=None
args=parser.parse_args(arguments.split())


In [4]:
args.output_dir=Path(args.output_dir)
args.output_dir.mkdir(parents=True, exist_ok=True)
args.checkpoint_dir=args.output_dir/"checkpoints"
args.checkpoint_dir.mkdir(parents=True, exist_ok=True)
args.last=args.checkpoint_dir/args.last
args.best=args.checkpoint_dir/args.best
print(f"{args.last=}")
print(f"{args.best=}")

device=torch.device(args.device) if (torch.cuda.is_available() and args.device=='cuda') else torch.device('cpu')
print(f"{device=}")

torch.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)

args.last=WindowsPath('D:/results/ucf101/video_mae/mini_finetune_distilled_small/checkpoints/last.pth')
args.best=WindowsPath('D:/results/ucf101/video_mae/mini_finetune_distilled_small/checkpoints/best.pth')
device=device(type='cuda')


In [5]:
args.nb_classes=101
model=vit_small_patch16_224(pretrained=False, img_size=args.input_size, num_classes=args.nb_classes, all_frames=args.num_frames*args.num_segments,
                          tubelet_size=args.tubelet_size, drop_rate=args.drop, drop_path_rate=args.drop_path, attn_drop_rate=args.attn_drop_rate,
                          head_drop_rate=args.head_drop_rate, use_mean_pooling=args.use_mean_pooling, init_scale=args.init_scale,
                          with_cp=args.with_checkpoint)#, cos_attn=args.cos_attn)
print(f"{args.tubelet_size=}, {args.drop=}, {args.drop_path=}, {args.attn_drop_rate=}, {args.head_drop_rate=}., {args.use_mean_pooling=}")
print(f"{args.init_scale=}, {args.with_checkpoint=}")
args.patch_size=model.patch_embed.patch_size
args.window_size=(args.num_frames//args.tubelet_size, args.input_size//args.patch_size[0], args.input_size//args.patch_size[1])
print(f"Patch size: {args.patch_size}, Number of patches per dim: {args.window_size}")


args.tubelet_size=2, args.drop=0.0, args.drop_path=0.3, args.attn_drop_rate=0.0, args.head_drop_rate=0.0., args.use_mean_pooling=True
args.init_scale=0.001, args.with_checkpoint=False
Patch size: (16, 16), Number of patches per dim: (8, 14, 14)


In [6]:
initial_model_params={name:param.clone() for name, param in model.named_parameters()}

In [7]:
if isinstance(args.finetune, str) and os.path.isfile(args.finetune):
    print(f"Pretrained model file, {args.finetune}, existence is {os.path.isfile(args.finetune)}")
    load_pretrained_model(args=args, model=model, weight_fpath=args.finetune)

Pretrained model file, D:\results\ucf101\video_mae\distilled_small\vit_s_k710_dl_from_giant.pth, existence is True

Weights of VisionTransformer not initialized from pretrained model: ['head.weight', 'head.bias']


In [8]:
for name, param in model.named_parameters():
    same=torch.allclose(param, initial_model_params[name])
    if same: print(f"{name} does not change")

head.weight does not change
head.bias does not change


In [9]:
for name, param in model.named_parameters():
    same=torch.allclose(param, initial_model_params[name])
    if not same: print(f"{name} does change")

patch_embed.proj.weight does change
patch_embed.proj.bias does change
blocks.0.norm1.weight does change
blocks.0.norm1.bias does change
blocks.0.attn.q_bias does change
blocks.0.attn.v_bias does change
blocks.0.attn.qkv.weight does change
blocks.0.attn.proj.weight does change
blocks.0.attn.proj.bias does change
blocks.0.norm2.weight does change
blocks.0.norm2.bias does change
blocks.0.mlp.fc1.weight does change
blocks.0.mlp.fc1.bias does change
blocks.0.mlp.fc2.weight does change
blocks.0.mlp.fc2.bias does change
blocks.1.norm1.weight does change
blocks.1.norm1.bias does change
blocks.1.attn.q_bias does change
blocks.1.attn.v_bias does change
blocks.1.attn.qkv.weight does change
blocks.1.attn.proj.weight does change
blocks.1.attn.proj.bias does change
blocks.1.norm2.weight does change
blocks.1.norm2.bias does change
blocks.1.mlp.fc1.weight does change
blocks.1.mlp.fc1.bias does change
blocks.1.mlp.fc2.weight does change
blocks.1.mlp.fc2.bias does change
blocks.2.norm1.weight does chang